In [ ]:
# Runner Configuration

PROJECT_WORKDIR = "/kaggle/temp/project"
SETUP_COMMAND = None
SOURCE_COMMIT = ""


In [ ]:
# Project Setup

import json
import shutil
import subprocess
import zipfile
from pathlib import Path

SOURCE_MARKER_NAME = "kaggle_runner_source.json"

project_workdir = Path(PROJECT_WORKDIR)

marker_matches = []

for marker_path in Path("/kaggle/input").rglob(
    SOURCE_MARKER_NAME
):
    try:
        marker = json.loads(
            marker_path.read_text(
                encoding="utf-8"
            )
        )
    except (json.JSONDecodeError, OSError):
        continue

    if marker.get("commit") == SOURCE_COMMIT:
        marker_matches.append(marker_path)

if len(marker_matches) > 1:
    raise RuntimeError(
        "Found multiple source roots for commit "
        f"{SOURCE_COMMIT}: {marker_matches}"
    )

if project_workdir.exists():
    if (
        project_workdir.is_symlink()
        or project_workdir.is_file()
    ):
        project_workdir.unlink()
    else:
        shutil.rmtree(project_workdir)

if len(marker_matches) == 1:
    source_path = marker_matches[0].parent

    shutil.copytree(
        source_path,
        project_workdir,
        ignore=shutil.ignore_patterns(
            SOURCE_MARKER_NAME
        ),
    )

    source_description = str(source_path)

else:
    archive_matches = []

    for archive_path in Path("/kaggle/input").rglob(
        "*.zip"
    ):
        try:
            with zipfile.ZipFile(
                archive_path,
                mode="r",
            ) as archive:
                if SOURCE_MARKER_NAME not in archive.namelist():
                    continue

                marker = json.loads(
                    archive.read(
                        SOURCE_MARKER_NAME
                    ).decode("utf-8")
                )

                if marker.get("commit") == SOURCE_COMMIT:
                    archive_matches.append(
                        archive_path
                    )
        except (
            zipfile.BadZipFile,
            UnicodeDecodeError,
            json.JSONDecodeError,
            OSError,
        ):
            continue

    if len(archive_matches) != 1:
        raise RuntimeError(
            "Could not resolve exactly one project source "
            f"for commit {SOURCE_COMMIT}. "
            f"markers={marker_matches}, "
            f"archives={archive_matches}"
        )

    project_workdir.mkdir(
        parents=True,
        exist_ok=False,
    )

    with zipfile.ZipFile(
        archive_matches[0],
        mode="r",
    ) as archive:
        archive.extractall(
            project_workdir
        )

    marker_inside_project = (
        project_workdir
        / SOURCE_MARKER_NAME
    )

    if marker_inside_project.exists():
        marker_inside_project.unlink()

    source_description = str(
        archive_matches[0]
    )

if not project_workdir.is_dir():
    raise RuntimeError(
        f"Project workdir was not created: {project_workdir}"
    )

if not any(project_workdir.iterdir()):
    raise RuntimeError(
        f"Project workdir is empty: {project_workdir}"
    )

if SETUP_COMMAND is not None and SETUP_COMMAND.strip():
    subprocess.run(
        [
            "bash",
            "-c",
            "set -e -o pipefail\n"
            + SETUP_COMMAND,
        ],
        check=True,
        cwd=project_workdir,
    )

print(f"Source loaded from: {source_description}")
print(f"Project workdir: {project_workdir}")

if SETUP_COMMAND is None or not SETUP_COMMAND.strip():
    print("No setup command configured.")
else:
    print("Setup completed.")


In [ ]:
# Job Definition

JOB_ID = ""
EXECUTION_ID = ""
SUBMITTED_AT = ""
WORKER_NUMBER = 0
COMMAND = ""


In [ ]:
# Job Execution

import json
import subprocess
from pathlib import Path

job_metadata = {
    "job_id": JOB_ID,
    "execution_id": EXECUTION_ID,
    "submitted_at": SUBMITTED_AT,
    "worker": WORKER_NUMBER,
    "source_commit": SOURCE_COMMIT,
    "command": COMMAND,
}

Path("/kaggle/working/job_metadata.json").write_text(
    json.dumps(job_metadata, indent=2),
    encoding="utf-8",
)

subprocess.run(
    ["bash", "-c", COMMAND],
    check=True,
    cwd=PROJECT_WORKDIR,
)
